# Day 5 · Gold and AI/BI expectations

Independent source-derived calculations; not a Lakehouse engine.



## Goal

Derive the final table contracts, reconcile the same 75 trips, and distinguish unknown future labels from zero. [Day 5 guide](README.md).



## 1. Verify inputs



In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset
require_fixed_dataset(SOURCE)
print("MASAR_SMALL_V1: original source hashes verified")

## 2. Build reference tables

The same corrected trips feed all consumers; no replacement dataset is introduced.



In [ ]:
from masar.serving_reference import day05_reference
reference = day05_reference(SOURCE)
print("Scope:", reference["scope"])
for table, count in reference["row_counts"].items():
    print(f"{table}: {count} rows")
assert reference["engine_executed"] is False

## 3. Read the Gold grain

Hourly rows count observed trip starts. Missing hours are not certified zero demand. Driver-day rows use the local start date, including the late trips crossing midnight.



In [ ]:
tables = reference["tables"]
print(json.dumps(tables["gold.zone_hourly_demand"][:3], indent=2))
print(json.dumps(tables["gold.driver_daily"][:3], indent=2))

## 4. Reconcile BI independently

The SQL check uses integer minor units to avoid floating-point money error. SQLite here is a read-only arithmetic cross-check, not the course storage engine.



In [ ]:
from masar.serving_reference import independent_sql_check
print(json.dumps(independent_sql_check(tables), indent=2))
assert reference["reconciliation"]["trip_count"] == 75
assert reference["reconciliation"]["fare_minor"] == 188060
assert reference["reconciliation"]["gps_events"] == 217

## 5. Inspect time-safe feature inputs

At scenario time 4 June 2026, 06:05 Riyadh time, use completed and already delivered history only. The next forecast hour starts at 07:00 Riyadh time.



In [ ]:
from masar.serving_reference import FEATURE_INPUTS, validate_feature_inputs
validate_feature_inputs(FEATURE_INPUTS)
print("Model-input allowlist:", FEATURE_INPUTS)
print(json.dumps(tables["ai.zone_hourly_features"], indent=2))
print(json.dumps(tables["ai.zone_hourly_labels"], indent=2))
assert all(row["target_trip_count"] is None for row in tables["ai.zone_hourly_labels"])

## 6. Check the arrival-time trap

These files were delivered on 4 June in the scenario. Looking back to 2 June does not make the later delivery available earlier. An empty observed history is not proof of zero real activity.



In [ ]:
from masar.delta_reference import day03_reference
from masar.trust_reference import events_from_source, GPS_FILES
from masar.serving_reference import tables_from_trusted
trips = day03_reference(SOURCE)["expected_corrected_rows"]
events = [event for name in GPS_FILES for event in events_from_source(SOURCE, name)]
early = tables_from_trusted(trips, events, SOURCE, as_of="2026-06-02T03:05:00Z")
assert all(not r["history_available"] for r in early["ai.zone_hourly_features"])
print("Early-cutoff history is unavailable for all three city proxies, not certified zero demand.")

## 7. Test two deliberately invalid outputs

Transient copies only: the original sources are never edited.



In [ ]:
from copy import deepcopy
from masar.serving_reference import validate_tables
for problem in ["duplicate_fact", "invented_zero_label"]:
    probe = deepcopy(tables)
    if problem == "duplicate_fact":
        probe["bi.fact_trips"].append(deepcopy(probe["bi.fact_trips"][0]))
    else:
        probe["ai.zone_hourly_labels"][0]["target_trip_count"] = 0
    try:
        validate_tables(probe)
    except ValueError as exc:
        print(problem, "REJECTED:", exc)
    else:
        raise AssertionError("Deliberate defect was incorrectly accepted")

## 8. Save the actual reference result

Compare these expectations to native readback in Labs 07/08. They cannot replace native execution evidence.



In [ ]:
from masar.workspace import new_workspace, workspace_path, write_json
assert all(reference["checks"].values())
work = new_workspace(ROOT, "day05_reference")
write_json(workspace_path(work, "reports/day05_reference.json"), reference)
print("Reference checks passed:", len(reference["checks"]))
print("New outputs/day05_reference_* workspace saved. No native engine execution claimed.")

## Next

[Lab 07 native integration](STUDENT.ipynb) → [Lab 08 serving](STUDENT.ipynb). Native prerequisites remain mandatory.

